# Module 7: Spectrograms -- Watching the Brain Change Over Time

This tutorial walks through the concepts step by step. Each section includes an explanation, an example, and an exercise.

## Step 1: Setup & Imports

This notebook covers **spectrograms** -- time-frequency representations that show you *when* each frequency appears in a signal.

An FFT tells you *what* frequencies are present (averaged over the whole recording). A spectrogram tells you *what* frequencies are present **and when**.

The recipe:
1. **Chop** the signal into short, overlapping time windows
2. **Compute the FFT** of each window
3. **Stack** the results: time on x-axis, frequency on y-axis, color = power

This is how neurologists track brain dynamics: sleep stages, seizure evolution, drug effects, and more.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq
%matplotlib inline

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

**Exercise 1:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 2: Build a Chirp Signal (Frequency Increases Over Time)

A **chirp** is a signal whose frequency changes over time -- perfect for demonstrating spectrograms. We build one that sweeps from 5 Hz to 40 Hz over 10 seconds. A standard FFT would just show a broad hump across all frequencies. A spectrogram shows the sweep.

In [ ]:
# Generate a chirp signal: frequency sweeps from 5 Hz to 40 Hz
np.random.seed(42)
fs = 512
duration = 10
t = np.arange(0, duration, 1/fs)

# Linear chirp
chirp_signal = signal.chirp(t, f0=5, t1=duration, f1=40, method='linear')
chirp_signal += 0.2 * np.random.randn(len(t))  # Add noise

# Plot time domain
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, chirp_signal, 'k-', linewidth=0.5)
ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Amplitude', fontsize=12)
ax.set_title('Chirp Signal: Frequency sweeps from 5 Hz to 40 Hz', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Notice: the oscillations get visibly faster as time progresses.')
print('A standard FFT would average all these frequencies together.')
print('We need a spectrogram to see the sweep.')

**Exercise 2:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 3: Step-by-Step STFT: Windowing and FFT at Each Position

The **Short-Time Fourier Transform (STFT)** is the algorithm behind spectrograms.

For each time window:
1. Extract a segment of the signal
2. Apply a **window function** (e.g., Hann window) to taper the edges
3. Compute the FFT of the windowed segment
4. Store the magnitude (power) spectrum

Let us walk through this for three selected windows.

In [ ]:
# Manual STFT: show what happens at three selected windows
win_sec = 1.0  # 1-second window
win_samples = int(win_sec * fs)

# Three windows: early (expect low freq), middle (mid freq), late (high freq)
window_starts = [1.0, 5.0, 8.5]  # seconds
window_colors = ['#2980b9', '#e67e22', '#8e44ad']
window_labels = ['Early (expect ~8 Hz)', 'Middle (expect ~22 Hz)', 'Late (expect ~37 Hz)']

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Top: signal with windows highlighted
axes[0].plot(t, chirp_signal, 'k-', linewidth=0.5, alpha=0.6)
for ws, color, label in zip(window_starts, window_colors, window_labels):
    axes[0].axvspan(ws, ws + win_sec, alpha=0.2, color=color, label=label)
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Signal with Three Selected Windows', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)

# Bottom: FFT of each window
hann = np.hanning(win_samples)
for ws, color, label in zip(window_starts, window_colors, window_labels):
    start_idx = int(ws * fs)
    segment = chirp_signal[start_idx:start_idx + win_samples]
    windowed = segment * hann
    
    # Compute FFT
    Y = fft(windowed)
    freqs = fftfreq(win_samples, 1/fs)[:win_samples//2]
    power = 2.0/win_samples * np.abs(Y[:win_samples//2])**2
    power_db = 10 * np.log10(power + 1e-10)
    
    axes[1].plot(freqs, power_db, color=color, linewidth=2, label=label)

axes[1].set_xlabel('Frequency (Hz)', fontsize=12)
axes[1].set_ylabel('Power (dB)', fontsize=12)
axes[1].set_title('FFT of Each Window', fontsize=13, fontweight='bold')
axes[1].set_xlim(0, 50)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print('Each window captures a different dominant frequency.')
print('The spectrogram stacks ALL these FFTs side by side.')

**Exercise 3:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 4: Plot the Spectrogram

Now we compute and plot the full spectrogram using `scipy.signal.spectrogram`. Each vertical slice is the FFT of one short time window. Together they show the chirp's frequency sweep from 5 Hz to 40 Hz.

In [ ]:
# Compute and plot the spectrogram of the chirp
nperseg = int(0.5 * fs)  # 0.5-second window
noverlap = int(0.4 * fs)  # 80% overlap

f, t_spec, Sxx = signal.spectrogram(chirp_signal, fs=fs,
                                     nperseg=nperseg,
                                     noverlap=noverlap,
                                     scaling='spectrum')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8),
                                gridspec_kw={'height_ratios': [1, 2]})

# Time domain on top
ax1.plot(t, chirp_signal, 'k-', linewidth=0.5)
ax1.set_ylabel('Amplitude')
ax1.set_title('Chirp Signal + Spectrogram', fontsize=14, fontweight='bold')
ax1.set_xlim(0, duration)

# Spectrogram on bottom
im = ax2.pcolormesh(t_spec, f, 10*np.log10(Sxx + 1e-10),
                     shading='gouraud', cmap='viridis')
ax2.set_ylabel('Frequency (Hz)', fontsize=12)
ax2.set_xlabel('Time (s)', fontsize=12)
ax2.set_ylim(0, 50)
plt.colorbar(im, ax=ax2, label='Power (dB)')

plt.tight_layout()
plt.show()

print('The bright diagonal line shows the frequency sweep from 5 Hz to 40 Hz.')
print('Each column is the FFT of one short window. Stacked together = spectrogram.')

**Exercise 4:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 5: The Window Size Tradeoff

Here is the fundamental tradeoff in time-frequency analysis -- almost like a Heisenberg uncertainty principle:

> **You cannot have perfect time resolution and perfect frequency resolution simultaneously.**

- **Small window** = good time resolution (sharp events) but poor frequency resolution (bands are blurry)
- **Large window** = good frequency resolution (clean bands) but poor time resolution (events are smeared)

The math: if your window is $T$ seconds, frequency resolution is $\Delta f = 1/T$ Hz.
- 0.25s window -> 4 Hz resolution
- 1.0s window -> 1 Hz resolution
- 2.0s window -> 0.5 Hz resolution

In [ ]:
# Demonstrate window size tradeoff
# Signal: abrupt switch from 10 Hz to 25 Hz at t=5s
np.random.seed(99)
fs = 512
dur = 10
t_demo = np.arange(0, dur, 1/fs)
freq_switch = np.where(t_demo < 5, 10, 25)
demo_signal = np.sin(2*np.pi*np.cumsum(freq_switch)/fs) + 0.2*np.random.randn(len(t_demo))

# Three different window sizes
window_sizes = [0.25, 1.0, 2.0]
titles = [
    'Short window (0.25s)\nGood time resolution, poor frequency resolution',
    'Medium window (1.0s)\nBalanced tradeoff',
    'Long window (2.0s)\nGood frequency resolution, poor time resolution'
]

fig, axes = plt.subplots(3, 1, figsize=(14, 12))
fig.suptitle('Window Size Tradeoff: Signal switches from 10 Hz to 25 Hz at t=5s',
             fontsize=14, fontweight='bold')

for ax, win_sec, title in zip(axes, window_sizes, titles):
    nperseg = int(win_sec * fs)
    noverlap = int(0.75 * nperseg)
    f, t_s, Sxx = signal.spectrogram(demo_signal, fs=fs,
                                      nperseg=nperseg,
                                      noverlap=noverlap)
    im = ax.pcolormesh(t_s, f, 10*np.log10(Sxx + 1e-10),
                        shading='gouraud', cmap='viridis')
    ax.axvline(5, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.set_ylim(0, 40)
    ax.set_ylabel('Freq (Hz)')
    ax.set_title(title, fontsize=11, loc='left')
    freq_res = 1/win_sec
    ax.text(0.98, 0.95, f'Freq resolution: {freq_res:.1f} Hz',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=10, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

axes[-1].set_xlabel('Time (s)', fontsize=12)
plt.tight_layout()
plt.show()

print('Short window: the transition at t=5s is sharp, but 10 Hz and 25 Hz bands are blurry.')
print('Long window: the frequency bands are crisp, but the transition is smeared.')
print('This tradeoff is unavoidable. Choose your window based on what matters most.')

**Exercise 5:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 6: Simulating Sleep Stages with Changing Frequency Bands

In clinical EEG, spectrograms are used to visualize **sleep staging**. As a patient falls asleep, the dominant frequency shifts:

- **Awake**: alpha (8-13 Hz) dominant
- **Stage N1** (drowsiness): theta (4-8 Hz) appears
- **Stage N2** (light sleep): sleep spindles (12-14 Hz) + K-complexes
- **Stage N3** (deep sleep): delta (1-4 Hz) dominant
- **REM**: mixed low-amplitude, theta-dominant

We simulate a simplified version of this progression.

In [ ]:
# Simulate sleep-like frequency evolution
np.random.seed(42)
fs = 256
# 5 stages, 10 seconds each = 50 seconds total
stage_duration = 10  # seconds per stage
stages = [
    {'name': 'Awake', 'freq': 10, 'amp': 30, 'color': '#8e44ad'},
    {'name': 'N1 (Drowsy)', 'freq': 6, 'amp': 40, 'color': '#1abc9c'},
    {'name': 'N2 (Light Sleep)', 'freq': 12, 'amp': 25, 'color': '#2980b9'},
    {'name': 'N3 (Deep Sleep)', 'freq': 2, 'amp': 100, 'color': '#2c3e50'},
    {'name': 'REM', 'freq': 5, 'amp': 20, 'color': '#e74c3c'},
]

total_duration = stage_duration * len(stages)
t_sleep = np.arange(0, total_duration, 1/fs)
sleep_signal = np.zeros(len(t_sleep))

for i, stage in enumerate(stages):
    start = i * stage_duration
    end = (i + 1) * stage_duration
    mask = (t_sleep >= start) & (t_sleep < end)
    # Main frequency + harmonics + noise
    sleep_signal[mask] = (stage['amp'] * np.sin(2*np.pi*stage['freq']*t_sleep[mask]) +
                          stage['amp']*0.3 * np.sin(2*np.pi*(stage['freq']*1.5)*t_sleep[mask]) +
                          8 * np.random.randn(mask.sum()))

# Compute spectrogram
nperseg = int(2 * fs)  # 2-second windows for sleep staging
noverlap = int(1.5 * fs)
f_sl, t_sl, Sxx_sl = signal.spectrogram(sleep_signal, fs=fs,
                                         nperseg=nperseg,
                                         noverlap=noverlap)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8),
                                gridspec_kw={'height_ratios': [1, 2]})

# Time domain with stage labels
ax1.plot(t_sleep, sleep_signal, 'k-', linewidth=0.5, alpha=0.7)
for i, stage in enumerate(stages):
    start = i * stage_duration
    ax1.axvspan(start, start + stage_duration, alpha=0.15, color=stage['color'])
    ax1.text(start + stage_duration/2, max(sleep_signal)*0.85, stage['name'],
             ha='center', fontsize=9, fontweight='bold', color=stage['color'])
ax1.set_ylabel('uV')
ax1.set_title('Simulated Sleep Progression', fontsize=14, fontweight='bold')
ax1.set_xlim(0, total_duration)

# Spectrogram
im = ax2.pcolormesh(t_sl, f_sl, 10*np.log10(Sxx_sl + 1e-10),
                     shading='gouraud', cmap='viridis')
ax2.set_ylabel('Frequency (Hz)', fontsize=12)
ax2.set_xlabel('Time (s)', fontsize=12)
ax2.set_ylim(0, 25)
for i in range(1, len(stages)):
    ax2.axvline(i*stage_duration, color='white', linestyle='--', linewidth=1, alpha=0.5)
plt.colorbar(im, ax=ax2, label='Power (dB)')

plt.tight_layout()
plt.show()

print('Each sleep stage has a different dominant frequency.')
print('The spectrogram makes this progression immediately visible.')

**Exercise 6:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 7: Exercise 1: Falling Asleep (Alpha to Delta)

**Your turn.** Create a signal that simulates falling asleep:

- First half: alpha rhythm (10 Hz) -- the patient is awake, eyes closed
- Second half: delta rhythm (2 Hz) -- the patient has entered deep sleep

Plot the signal and its spectrogram. You should see the bright band shift from 10 Hz down to 2 Hz.

**Exercise 7:** Try it yourself!

In [ ]:
# EXERCISE 1: Alpha to Delta transition (falling asleep)
# =======================================================
# 1. Create a 10-second signal at fs=256
#    - First 5 seconds: 10 Hz (alpha), amplitude 30 uV
#    - Last 5 seconds: 2 Hz (delta), amplitude 80 uV
# 2. Add some noise for realism
# 3. Compute the spectrogram (try nperseg = 2*fs)
# 4. Plot both signal and spectrogram

# YOUR CODE HERE
fs = 256
duration = 10
t_ex = np.arange(0, duration, 1/fs)

# falling_asleep = ???

# Hint: use np.where(t_ex < 5, ..., ...) to switch frequencies
# Then use signal.spectrogram() and plt.pcolormesh() to visualize

## Step 8: Exercise 2: Window Size Experiment

**Your turn.** Using the chirp signal from earlier, compute spectrograms with three different window sizes and observe the tradeoff.

**Questions to answer:**
- Which window size best resolves the exact frequency at each moment?
- Which window size best shows the precise timing of changes?
- What window size would you choose for seizure detection? For sleep staging?

**Exercise 8:** Try it yourself!

In [ ]:
# EXERCISE 2: Window size experiment
# ====================================
# Use the chirp signal (chirp_signal) defined earlier
# Compute spectrograms with windows of: 0.1s, 0.5s, 2.0s
# Plot all three and compare

# YOUR CODE HERE
window_sizes_ex = [0.1, 0.5, 2.0]

# For each window size:
# nperseg = int(win * fs)
# f, t_s, Sxx = signal.spectrogram(chirp_signal, fs=fs, nperseg=nperseg)
# Plot using pcolormesh

# Answer these:
# 1. 0.1s window: freq resolution = 10 Hz. Can you even see the chirp?
# 2. 0.5s window: freq resolution = 2 Hz. Decent compromise.
# 3. 2.0s window: freq resolution = 0.5 Hz. Sharp bands, but smeared in time.
#
# Clinical guidance:
# - Seizure detection: 0.5-1s windows (need to catch fast changes)
# - Sleep staging: 2-5s windows (frequency precision matters more)

## Step 9: Clinical Applications

Spectrograms are used daily in clinical neuroscience:

| Application | What You See | Typical Window |
|-------------|-------------|----------------|
| **Seizure detection** | Evolving frequency pattern | 0.5-1s |
| **Sleep staging** | Slow frequency progression | 2-5s |
| **Anesthesia monitoring** | Frequency shift with drug effect | 1-2s |
| **Brain-computer interfaces** | Motor imagery bands | 0.25-0.5s |
| **ICU monitoring** | Long-term trends | 1-5s |

**Key takeaway:** A spectrogram is a stack of FFTs. Each column is the frequency content of one short time window. Together they show how the brain's frequency mix changes over time.

In [ ]:
# Summary: the time-frequency tradeoff
print('Time-Frequency Tradeoff Summary')
print('=' * 55)
print(f"{'Window (s)':<14} {'Freq Res (Hz)':<16} {'Best For'}")
print('-' * 55)
examples = [
    (0.25, 4.0, 'Fast events (e.g., BCIs)'),
    (0.50, 2.0, 'Seizure detection'),
    (1.00, 1.0, 'General purpose'),
    (2.00, 0.5, 'Sleep staging'),
    (5.00, 0.2, 'ICU long-term trends'),
]
for win, res, use in examples:
    print(f"{win:<14.2f} {res:<16.1f} {use}")

print('\nRemember: Frequency resolution = 1 / window_size')
print('You CANNOT improve both time and frequency resolution simultaneously.')

**Exercise 9:** Try it yourself!

In [ ]:
# YOUR CODE HERE
